# 04.5 — UK Schema Fetch

Fetches live HTML for `.co.uk` / `.uk` domains that already have schema.org markup in WDC,
then writes `data/raw/warc_manifest_uk.jsonl` in the same format as the `.ie` manifest.

**Inputs:**
- `data/raw/wdc_global_records.jsonl` — 100K global WDC records (produced by notebook 02)

**Outputs:**
- `data/raw/html_uk/` — fetched HTML files
- `data/raw/warc_manifest_uk.jsonl` — (url, html_path, wdc_jsonld) for UK pages

Notebook 05 automatically picks up this manifest when it exists.

**Resume-safe:** re-running skips already-fetched HTML files.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

print('Imports OK')

In [ ]:
## Config

GLOBAL_RECORDS = Path('../data/raw/wdc_global_records.jsonl')
HTML_DIR       = Path('../data/raw/html_uk')
OUT_MANIFEST   = Path('../data/raw/warc_manifest_uk.jsonl')

THREADS = 20
TIMEOUT = 15  # seconds per request
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; SchemaBot/1.0)'}

HTML_DIR.mkdir(parents=True, exist_ok=True)
print(f'HTML dir: {HTML_DIR}')

## Step 1 — Extract UK Records from WDC Global

In [ ]:
uk_records = []
with GLOBAL_RECORDS.open() as f:
    for line in f:
        r = json.loads(line)
        url = r.get('source_url', '')
        # .co.uk covers most UK businesses; .uk covers newer registrations
        if '.co.uk' in url or (url.endswith('.uk') and '.co.uk' not in url):
            uk_records.append(r)

print(f'UK records found: {len(uk_records)}')

# Schema type breakdown
from collections import Counter
types = Counter()
for r in uk_records:
    t = r.get('schema_type', 'Unknown')
    types[t] += 1
print('\nSchema type distribution:')
for t, c in types.most_common():
    print(f'  {t}: {c}')

## Step 2 — Fetch HTML (resume-safe)

In [ ]:
def _url_to_stem(url: str) -> str:
    """Deterministic filename from URL — same logic as fetch_uk.py."""
    return re.sub(r'[^\w]', '_', url)[:120]

already = {p.stem for p in HTML_DIR.glob('*.html')}
todo    = [r for r in uk_records if _url_to_stem(r['source_url']) not in already]

print(f'Already fetched : {len(already)}')
print(f'To fetch now    : {len(todo)}')

In [ ]:
def fetch_one(record):
    url  = record['source_url']
    stem = _url_to_stem(url)
    out  = HTML_DIR / f'{stem}.html'
    try:
        r = requests.get(url, timeout=TIMEOUT, headers=HEADERS, allow_redirects=True)
        if r.ok and len(r.text) > 200:
            out.write_text(r.text, encoding='utf-8', errors='replace')
            return {
                'url':            url,
                'html_path':      str(out),
                'wdc_jsonld':     record.get('jsonld'),
                'screenshot_path': None,
                'source':         'wdc_uk',
            }
    except Exception:
        pass
    return None

results = []
failures = 0

print(f'Fetching {len(todo)} pages ({THREADS} threads)...')
with ThreadPoolExecutor(max_workers=THREADS) as ex:
    futs = {ex.submit(fetch_one, r): r for r in todo}
    for f in tqdm(as_completed(futs), total=len(todo), desc='UK pages'):
        res = f.result()
        if res:
            results.append(res)
        else:
            failures += 1

print(f'\nFetched : {len(results)}')
print(f'Failed  : {failures}')

## Step 3 — Write Manifest

In [ ]:
# Merge newly fetched with anything already in the manifest
existing = []
if OUT_MANIFEST.exists():
    with OUT_MANIFEST.open() as f:
        existing = [json.loads(l) for l in f]
    print(f'Existing manifest entries: {len(existing)}')

# Dedup by url
seen_urls = {e['url'] for e in existing}
new_entries = [r for r in results if r['url'] not in seen_urls]
all_entries = existing + new_entries

with OUT_MANIFEST.open('w') as f:
    for e in all_entries:
        f.write(json.dumps(e, ensure_ascii=False) + '\n')

print(f'Manifest written: {len(all_entries)} entries → {OUT_MANIFEST}')
print('\nNext step: re-run notebook 05 (training data prep) — it picks up the UK manifest automatically.')